###  Unity Catalog Governance
#### Row Level Security (RLS) + Column Level Security (CLS)

#### Objective
This notebook implements **enterprise-grade governance controls** on the Gold layer tables using **Unity Catalog**.


Therefore, this project simulates real access control using:

- **Databricks account groups** (e.g. store managers, finance)
- **Unity Catalog Row Filters** for Row Level Security (RLS)
- **Unity Catalog Column Masks** for Column Level Security (CLS)

---

#### Security Strategy

#### Row Level Security (RLS)
Restrict data visibility by `store_id`:
- Store managers can only see their assigned store(s)
- Finance team can see all stores

Applied on:
- `coffee.gold.fact_transactions`
- `coffee.gold.fact_transaction_items` (derived via transaction_id → store_id)
- `coffee.gold.dim_stores`

#### Column Level Security (CLS)
Mask sensitive customer attributes:
- `dim_users.birthdate`
- `dim_users.gender`

Finance team can see full values, other users get masked values.


In [0]:
-- ==========================================================
-- Step 1: Create a dedicated schema for security objects
-- ==========================================================
CREATE SCHEMA IF NOT EXISTS coffee.security;


In [0]:
-- ==========================================================
-- Step 2: Create a mapping table for store access
-- This table simulates IAM role mapping:
-- which group is allowed to access which store_id.
-- ==========================================================

CREATE TABLE IF NOT EXISTS coffee.security.store_access (
  group_name STRING,
  store_id   INT
);


In [0]:
-- ==========================================================
-- Step 3: Load store access mappings
-- Dataset contains 10 stores (store_id 1-10).
-- ==========================================================

TRUNCATE TABLE coffee.security.store_access;

INSERT INTO coffee.security.store_access VALUES
('coffee_store_1_manager',  1),
('coffee_store_2_manager',  2),
('coffee_store_3_manager',  3),
('coffee_store_4_manager',  4),
('coffee_store_5_manager',  5),
('coffee_store_6_manager',  6),
('coffee_store_7_manager',  7),
('coffee_store_8_manager',  8),
('coffee_store_9_manager',  9),
('coffee_store_10_manager', 10);


In [0]:
-- ==========================================================
-- Step 4: RLS function for store-level filtering
--
-- Rules:
-- 1) Finance users see all stores
-- 2) Store managers see only their mapped store_id
-- ==========================================================
CREATE OR REPLACE FUNCTION coffee.security.fn_store_rls(store_id INT)
RETURN
  is_account_group_member('Coffee_finance')
  OR store_id IN (
    SELECT store_id
    FROM coffee.security.store_access
    WHERE is_account_group_member(group_name)
  );



In [0]:
-- ==========================================================
-- Step 5: Apply RLS on fact_transactions
-- ==========================================================
ALTER TABLE coffee.gold.fact_transactions
SET ROW FILTER coffee.security.fn_store_rls ON (store_id);


In [0]:
-- ==========================================================
-- Step 6: Apply RLS on dim_stores
-- ==========================================================
ALTER TABLE coffee.gold.dim_stores
SET ROW FILTER coffee.security.fn_store_rls ON (store_id);


In [0]:
-- ==========================================================
-- Step 7: Apply RLS on fact_transaction_items
-- ==========================================================
ALTER TABLE coffee.gold.fact_transaction_items
SET ROW FILTER coffee.security.fn_store_rls ON (store_id);


In [0]:
-- ==========================================================
-- Step 8: Mask birthdate for non-finance users
-- ==========================================================
CREATE OR REPLACE FUNCTION coffee.security.mask_birthdate(birthdate DATE)
RETURN
  CASE
    WHEN is_account_group_member('Coffee_finance') THEN birthdate
    ELSE NULL
  END;


In [0]:
-- ==========================================================
-- Step 9: Mask gender for non-finance users
-- ==========================================================
CREATE OR REPLACE FUNCTION coffee.security.mask_gender(gender STRING)
RETURN
  CASE
    WHEN is_account_group_member('Coffee_finance') THEN gender
    ELSE 'REDACTED'
  END;


In [0]:
-- ==========================================================
-- Step 10: Apply CLS on dim_users
-- ==========================================================
ALTER TABLE coffee.gold.dim_users
ALTER COLUMN birthdate
SET MASK coffee.security.mask_birthdate;

ALTER TABLE coffee.gold.dim_users
ALTER COLUMN gender
SET MASK coffee.security.mask_gender;


In [0]:
GRANT SELECT ON TABLE coffee.gold.fact_transactions TO `coffee_store_1_manager`;
GRANT SELECT ON TABLE coffee.gold.fact_transaction_items TO `coffee_store_1_manager`;
GRANT SELECT ON TABLE coffee.gold.dim_stores TO `coffee_store_1_manager`;
GRANT SELECT ON TABLE coffee.gold.dim_users TO `coffee_store_1_manager`;


In [0]:
GRANT USE CATALOG ON CATALOG coffee TO `coffee_store_1_manager`;
GRANT USE CATALOG ON CATALOG coffee TO `Coffee_finance`;


In [0]:
GRANT USE SCHEMA ON SCHEMA coffee.gold TO `coffee_store_1_manager`;
GRANT USE SCHEMA ON SCHEMA coffee.security TO `coffee_store_1_manager`;

GRANT USE SCHEMA ON SCHEMA coffee.gold TO `Coffee_finance`;
GRANT USE SCHEMA ON SCHEMA coffee.security TO `Coffee_finance`;


In [0]:
GRANT SELECT ON TABLE coffee.gold.fact_transactions TO `coffee_store_1_manager`;
GRANT SELECT ON TABLE coffee.gold.fact_transaction_items TO `coffee_store_1_manager`;
GRANT SELECT ON TABLE coffee.gold.dim_stores TO `coffee_store_1_manager`;
GRANT SELECT ON TABLE coffee.gold.dim_users TO `coffee_store_1_manager`;
